In [21]:
# Imports and Setup
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [22]:
#Load and Explore Data
data = pd.read_csv('../data/updated_ckd_dataset_with_stages.csv')

print(f"Dataset shape: {data.shape}")
print(f"\nFirst 5 rows:")
print(data.head())

Dataset shape: (4000, 23)

First 5 rows:
   serum_creatinine         gfr         bun  serum_calcium  ana       c3_c4  \
0          0.683683   32.946784    7.553739      10.039896    0  138.204989   
1          3.809044   32.685035  141.347494       8.330543    1   24.282343   
2          1.143827    2.079805   15.979104       9.419229    0  163.970666   
3          4.804657  109.871407   53.307333       7.556631    1   71.056846   
4          4.920235   42.214590  134.182157       7.289379    1   23.384639   

   hematuria  oxalate_levels  urine_ph  blood_pressure  ... smoking  \
0          0        2.878164  7.864308      115.224217  ...     yes   
1          1        4.767639  4.920015      130.143900  ...     yes   
2          0        1.818613  6.188115       98.026072  ...      no   
3          1        4.051686  5.278607      142.166650  ...      no   
4          1        3.240920  4.862923      151.962572  ...      no   

        alcohol  painkiller_usage family_history weight_c

In [23]:
# Check Target Distribution
target_col = 'ckd_pred'
print(f"Target Distribution ({target_col}):")
print(data[target_col].value_counts().sort_index())

Target Distribution (ckd_pred):
ckd_pred
CKD       3875
No CKD     125
Name: count, dtype: int64


In [24]:
#Train/Test Split
x = data.drop([target_col, 'ckd_stage', 'cluster'], axis=1, errors='ignore')
y = data[target_col]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training: {len(y_train)} samples")
print(f"Test: {len(y_test)} samples")

Training: 3200 samples
Test: 800 samples


In [25]:
# Create Client Data
num_clients = 4
client_data = []

train_data = x_train.copy()
train_data['Target'] = y_train
train_data = train_data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

base_size = len(train_data) // num_clients
remainder = len(train_data) % num_clients

start = 0
for i in range(num_clients):
    end = start + base_size + (1 if i < remainder else 0)
    client_subset = train_data.iloc[start:end]
    
    x_c = client_subset.drop('Target', axis=1)
    y_c = client_subset['Target']
    
    client_data.append((x_c, y_c))
    start = end

print(f"Created {num_clients} clients:")
for i, (x_c, y_c) in enumerate(client_data, start=1):
    print(f"Client {i}: {x_c.shape[0]} samples | Classes: {y_c.value_counts().to_dict()}")

Created 4 clients:
Client 1: 800 samples | Classes: {'CKD': 770, 'No CKD': 30}
Client 2: 800 samples | Classes: {'CKD': 778, 'No CKD': 22}
Client 3: 800 samples | Classes: {'CKD': 770, 'No CKD': 30}
Client 4: 800 samples | Classes: {'CKD': 782, 'No CKD': 18}
